<a href="https://colab.research.google.com/github/Sanjay-iiot/HW4/blob/main/Assignment4_Part2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import subprocess
subprocess.run(["pip", "install", "tiktoken", "-q"])

import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken, itertools, pandas as pd, time, math

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cuda


In [6]:
subprocess.run(["wget","-q","-nc",
    "https://raw.githubusercontent.com/TTAyanlade/vlm/main/tips_ML_Phenotype.txt"])

with open('tips_ML_Phenotype.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(f"Characters: {len(text):,}")
print(f"Sample: {text[:100]}")

Characters: 36,358
Sample: Phenotyping Data and ML
The enormous volume, variety, velocity, and veracity of imaging and remote-s


In [3]:
#a): tiktoken — demonstrate and report the effect

In [8]:
enc_tiktoken = tiktoken.get_encoding("gpt2")
tik_tokens   = enc_tiktoken.encode(text)

# Original char-level (from notebook)
chars      = sorted(list(set(text)))
char_vocab = len(chars)
stoi       = {ch: i for i, ch in enumerate(chars)}
itos       = {i: ch for i, ch in enumerate(chars)}
char_encode = lambda s: [stoi[c] for c in s]
char_decode = lambda l: ''.join([itos[i] for i in l])



# Use char-level for training (stable on this dataset size)
data       = torch.tensor(char_encode(text), dtype=torch.long)
vocab_size = char_vocab
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f"  Training vocab: {vocab_size} chars | Train: {len(train_data):,} | Val: {len(val_data):,}")

  Training vocab: 81 chars | Train: 32,722 | Val: 3,636


In [5]:
#b): RoPE

In [10]:
def rope_freqs(head_dim, seq_len, base=10000.0):
    half  = head_dim // 2
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang   = torch.outer(torch.arange(seq_len).float(), theta)
    return torch.cos(ang), torch.sin(ang)

def apply_rope(x, cos_t, sin_t):
    B, T, D = x.shape
    half    = D // 2
    x1, x2  = x[..., :half], x[..., half:]
    c = cos_t[:T].unsqueeze(0)
    s = sin_t[:T].unsqueeze(0)
    return torch.cat([x1*c - x2*s, x1*s + x2*c], dim=-1)

print(f"  b): RoPE vs Absolute Position Embedding")
print(f"  apply_rope(q, cos_t, sin_t) and apply_rope(k, cos_t, sin_t)")
print(f"  inside every attention head — position encoded at every layer")

  b): RoPE vs Absolute Position Embedding
  apply_rope(q, cos_t, sin_t) and apply_rope(k, cos_t, sin_t)
  inside every attention head — position encoded at every layer


In [20]:
#Module
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout, cos_t, sin_t):
        super().__init__()
        self.head_size = head_size
        self.q    = nn.Linear(n_embd, head_size, bias=False)
        self.k    = nn.Linear(n_embd, head_size, bias=False)
        self.v    = nn.Linear(n_embd, head_size, bias=False)
        self.drop = nn.Dropout(dropout)
        self.register_buffer('tril',  torch.tril(torch.ones(block_size, block_size)))
        self.register_buffer('cos_t', cos_t)
        self.register_buffer('sin_t', sin_t)

    def forward(self, x):
        B, T, _ = x.shape
        q = apply_rope(self.q(x), self.cos_t, self.sin_t)
        k = apply_rope(self.k(x), self.cos_t, self.sin_t)
        v = self.v(x)
        w = (q @ k.transpose(-2,-1)) * (self.head_size ** -0.5)
        w = w.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        w = self.drop(F.softmax(w, dim=-1))
        return w @ v

class MHA(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout, cos_t, sin_t):
        super().__init__()
        hs = n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, hs, block_size, dropout, cos_t, sin_t)
            for _ in range(n_head)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class FFN(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd), nn.ReLU(),
            nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout, cos_t, sin_t):
        super().__init__()
        self.sa  = MHA(n_embd, n_head, block_size, dropout, cos_t, sin_t)
        self.ffn = FFN(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class LM(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.2):
        super().__init__()
        self.block_size = block_size
        self.emb    = nn.Embedding(vocab_size, n_embd)
        c, s = rope_freqs(n_embd // n_head, block_size)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout, c, s)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        x      = self.emb(idx)
        x      = self.blocks(x)
        x      = self.ln_f(x)
        logits = self.head(x)
        loss   = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            ic        = idx[:, -self.block_size:]
            logits, _ = self(ic)
            probs     = F.softmax(logits[:,-1,:], dim=-1)
            idx       = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

In [21]:
#Helpers
def get_batch(split, bs=16, bl=32):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - bl, (bs,))
    x  = torch.stack([d[i:i+bl]     for i in ix]).to(device)
    y  = torch.stack([d[i+1:i+bl+1] for i in ix]).to(device)
    return x, y

@torch.no_grad()
def est_loss(model, iters=200, bs=16, bl=32):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        ls = torch.zeros(iters)
        for k in range(iters):
            x, y    = get_batch(split, bs, bl)
            _, loss = model(x, y)
            ls[k]   = loss.item()
        out[split] = ls.mean().item()
    model.train()
    return out

In [9]:
# MODIFICATION (c): Hyperparameter grid search
# Same settings as original notebook: batch=16, block=32, lr=1e-3
# n_layer ∈ {4, 6, 8}  ×  n_embd ∈ {64, 128, 256}

In [24]:
BATCH_SIZE    = 16
BLOCK_SIZE    = 32
MAX_ITERS     = 5000
EVAL_INTERVAL = 500
LR            = 1e-3
N_HEAD        = 4
DROPOUT       = 0.2

N_LAYERS = [4, 6, 8]
N_EMBDS  = [64, 128, 256]

print(f"  MODIFICATION (c): Hyperparameter Grid Search")
print(f"  {len(N_LAYERS)*len(N_EMBDS)} configs | char-level vocab={vocab_size}")
print(f"  Baseline (original, n_layer=4, n_embd=64): val=2.0393")

results = []

for n_layer, n_embd in itertools.product(N_LAYERS, N_EMBDS):
    print(f"\n>>> n_layer={n_layer}  n_embd={n_embd}")

    model = LM(vocab_size, n_embd, N_HEAD, n_layer, BLOCK_SIZE, DROPOUT).to(device)
    np_   = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"    Params: {np_:.3f}M")
    print(f"    {'Iter':>6}  {'Train':>8}  {'Val':>8}")
    print(f"    {'-'*26}")

    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    t0  = time.time()

    for i in range(MAX_ITERS):
        if i % EVAL_INTERVAL == 0 or i == MAX_ITERS - 1:
            ls = est_loss(model, 200, BATCH_SIZE, BLOCK_SIZE)
            print(f"    {i:>6}  {ls['train']:>8.4f}  {ls['val']:>8.4f}")
        xb, yb  = get_batch('train', BATCH_SIZE, BLOCK_SIZE)
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    elapsed = time.time() - t0
    final   = est_loss(model, 200, BATCH_SIZE, BLOCK_SIZE)

    model.eval()
    ctx  = torch.zeros((1,1), dtype=torch.long, device=device)
    samp = char_decode(model.generate(ctx, 200)[0].tolist())
    print(f"\n    Sample: {samp[:250]}")

    results.append({
        'n_layer':    n_layer,
        'n_embd':     n_embd,
        'Params(M)':  round(np_, 3),
        'Train Loss': round(final['train'], 4),
        'Val Loss':   round(final['val'],   4),
        'Time(s)':    round(elapsed, 1)
    })
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()
 # Results
df   = pd.DataFrame(results).sort_values('Val Loss').reset_index(drop=True)
best = df.iloc[0]

print("  FINAL RESULTS — sorted by Val Loss (lower = better)")
print(df.to_string(index=False))

  MODIFICATION (c): Hyperparameter Grid Search
  9 configs | char-level vocab=81
  Baseline (original, n_layer=4, n_embd=64): val=2.0393

>>> n_layer=4  n_embd=64
    Params: 0.210M
      Iter     Train       Val
    --------------------------
         0    4.3941    4.3928
       500    2.0061    2.0735
      1000    1.7062    1.8578
      1500    1.5318    1.7589
      2000    1.4465    1.6881
      2500    1.3532    1.6412
      3000    1.3030    1.6548
      3500    1.2494    1.6090
      4000    1.2111    1.6318
      4500    1.1855    1.5999
      4999    1.1604    1.5985

    Sample: 
(B-neans, ECeniﬁcation
detection (sempora compared by from back as
ML was user assopization, classiﬁcation and suchowas, dimera image (SVM) and features was used brance of the (diseases of a various, 

>>> n_layer=4  n_embd=128
    Params: 0.813M
      Iter     Train       Val
    --------------------------
         0    4.4332    4.4393
       500    1.8388    1.9509
      1000    1.5301    1.7687